# Assignment 1 walkthrough: diabetes hospital encounters

Run the cells from top to bottom. Each step computes its result from `diabetic_data.csv`. Parts 1–4 are the design. Parts 5–7 build the same design in a SQLite database at `work/hospital.db` and write the MySQL scripts the submission asks for into `work/sql/`.

SQLite is what runs inside this notebook, so you can see the load immediately. The class dialect is MySQL. The `.sql` files are MySQL. The table names, keys, and inserts are the same design in both.

**Kernel:** the project virtual environment (`.venv`), which has pandas. From `hospital-readmissions`:

```bash
source .venv/bin/activate
cd work
jupyter notebook factcheck.ipynb
```

If Cursor asks for an interpreter, choose `.venv/bin/python`.

| Part | Steps | What you compute |
|---|---|---|
| 1 | 1.1–1.9 | File, attributes, keys, missing values, values that change across stays |
| 2 | 2.1–2.4 | Original relation, 1NF, partial dependencies, 2NF tables |
| 3 | 3.1–3.4 | Cardinality, participation, ER diagram |
| 4 | 4.1–4.4 | Diagram to foreign keys and creation order |
| 5 | 5.1–5.2 | Data types and `CREATE TABLE` |
| 6 | 6.1–6.5 | CSV to staging to the seven tables |
| 7 | 7.1–7.6 | Counts, keys, referential integrity, preserved information |


In [1]:
from pathlib import Path
import csv
import io
import sqlite3

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 88)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 120)

INT_LIMIT = 2_147_483_647  # largest MySQL signed INT

IGNORE = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "examide", "citoglipton", "insulin", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone",
    "change",
]

# Meanings from the dataset documentation (Strack et al. / the column guide).
# The 24 ignored columns are medication flags the assignment says to leave out.
DESCRIPTIONS = {
    "encounter_id": "Unique ID of one hospital stay",
    "patient_nbr": "ID of the patient. One patient can have several stays",
    "race": "Caucasian, AfricanAmerican, Hispanic, Asian, or Other",
    "gender": "Female, Male, or Unknown/Invalid",
    "age": "Age band in 10-year groups, such as [70-80)",
    "weight": "Weight band in pounds",
    "admission_type_id": "Code for the kind of admission (emergency, elective, and so on)",
    "discharge_disposition_id": "Code for where the patient went at discharge",
    "admission_source_id": "Code for where the patient was admitted from",
    "time_in_hospital": "Days in the hospital, from 1 to 14",
    "payer_code": "Insurance payer code, such as MC for Medicare",
    "medical_specialty": "Specialty of the admitting physician",
    "num_lab_procedures": "Lab tests during the stay",
    "num_procedures": "Non-lab procedures during the stay",
    "num_medications": "Distinct medications given during the stay",
    "number_outpatient": "Outpatient visits in the year before this stay",
    "number_emergency": "Emergency visits in the year before this stay",
    "number_inpatient": "Inpatient visits in the year before this stay",
    "diag_1": "Primary diagnosis, an ICD-9 code",
    "diag_2": "Secondary diagnosis, an ICD-9 code",
    "diag_3": "Additional diagnosis, an ICD-9 code",
    "number_diagnoses": "How many diagnoses were recorded for the stay",
    "max_glu_serum": "Glucose test result: >200, >300, Norm, or None if not measured",
    "A1Cresult": "HbA1c result: >7, >8, Norm, or None if not measured",
    "diabetesMed": "Whether any diabetes medication was prescribed: Yes or No",
    "readmitted": "NO, >30 (after 30 days), or <30 (within 30 days)",
}
for column in IGNORE:
    kind = "whether any medication dose changed" if column == "change" else f"the drug {column}"
    DESCRIPTIONS[column] = f"Ignored, as the assignment instructs. Records {kind}."

INT_COLUMNS = [
    "encounter_id", "patient_nbr", "admission_type_id", "discharge_disposition_id",
    "admission_source_id", "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency", "number_inpatient",
    "number_diagnoses",
]


def find_project():
    """Locate hospital-readmissions whether the kernel started in work/ or above it."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        data = folder / "data" / "diabetic_data.csv"
        if data.exists() and (folder / "work").is_dir():
            return folder
    raise FileNotFoundError(
        "Could not find data/diabetic_data.csv. Start the notebook from work/ "
        "or from hospital-readmissions/."
    )


ROOT = find_project()
DATA_PATH = ROOT / "data" / "diabetic_data.csv"
MAP_PATH = ROOT / "data" / "IDS_mapping.csv"
DB_PATH = ROOT / "work" / "hospital.db"
SQL_DIR = ROOT / "work" / "sql"

df = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
missing_docs = [c for c in df.columns if c not in DESCRIPTIONS]
if missing_docs:
    raise RuntimeError(f"No description for {missing_docs}")


def load_id_maps(path):
    """IDS_mapping.csv holds three lookup tables, separated by a line that is just a comma."""
    chunks, buf = [], []
    for line in Path(path).read_text().splitlines(keepends=True):
        if line.strip() in {"", ","}:
            if buf:
                chunks.append("".join(buf))
                buf = []
            continue
        buf.append(line)
    if buf:
        chunks.append("".join(buf))
    maps = {}
    for chunk in chunks:
        rows = list(csv.reader(io.StringIO(chunk)))
        key = rows[0][0]
        table = {}
        for row in rows[1:]:
            if not row or not row[0].strip():
                continue
            desc = row[1].strip() if len(row) > 1 else ""
            table[row[0].strip()] = None if desc in {"", "NULL"} else desc
        maps[key] = table
    return maps


ID_MAPS = load_id_maps(MAP_PATH)
for expected in ("admission_type_id", "discharge_disposition_id", "admission_source_id"):
    if expected not in ID_MAPS:
        raise RuntimeError(f"IDS_mapping.csv has no section for {expected}")


def banner(step, title):
    print(f"\n{'=' * 72}\nStep {step}   {title}\n{'=' * 72}")


def mysql_int(series):
    values = series.astype("int64")
    widest = max(abs(int(values.min())), abs(int(values.max())))
    return "BIGINT" if widest > INT_LIMIT else "INT"


def mysql_varchar(series, null_tokens=()):
    kept = series[~series.isin(set(null_tokens))]
    observed = int(kept.str.len().max()) if len(kept) else 1
    for size in (10, 20, 40, 80, 160, 255):
        if observed <= size:
            return f"VARCHAR({size})", observed
    return f"VARCHAR({observed})", observed


def sql_literal(value):
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"


def quote_sqlite(name):
    return '"' + name.replace('"', '""') + '"'


print(f"Project:  {ROOT}")
print(f"CSV:      {DATA_PATH.name}   shape {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Mapping:  {MAP_PATH.name}   sections {list(ID_MAPS)}")
print(f"Database: {DB_PATH}")


Project:  /Users/miaelena/Desktop/mia/HunterCS/49376/assignments/assignment1/hospital-readmissions
CSV:      diabetic_data.csv   shape 101,766 rows x 50 columns
Mapping:  IDS_mapping.csv   sections ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
Database: /Users/miaelena/Desktop/mia/HunterCS/49376/assignments/assignment1/hospital-readmissions/work/hospital.db


## Part 1. Select and examine the original dataset

The assignment asks for eight facts before any tables are designed. Steps 1.1–1.8 are those eight. Step 1.9 is the extra check that decides whether race, gender, and age belong to the patient or to the stay.


In [2]:
banner("1.1", "Dataset selected")
print("Option 2: Diabetes 130-US Hospitals for Years 1999-2008")
print("One row is one hospital stay (an encounter), not one patient.")

banner("1.2", "CSV file used")
print(DATA_PATH.name)

banner("1.3", "Total rows")
print(f"{len(df):,}")

banner("1.4", "Total columns")
used = [c for c in df.columns if c not in IGNORE]
print(f"{df.shape[1]} columns in the file")
print(f"{len(IGNORE)} ignored, as the assignment instructs")
print(f"{len(used)} columns enter the design")



Step 1.1   Dataset selected
Option 2: Diabetes 130-US Hospitals for Years 1999-2008
One row is one hospital stay (an encounter), not one patient.

Step 1.2   CSV file used
diabetic_data.csv

Step 1.3   Total rows
101,766

Step 1.4   Total columns
50 columns in the file
24 ignored, as the assignment instructs
26 columns enter the design


In [3]:
banner("1.5-1.6", "Every attribute, what it means, and what the file actually contains")

rows = []
for name in df.columns:
    series = df[name]
    known = series[~series.isin({"?", ""})]
    examples = ", ".join(known.drop_duplicates().head(3))
    rows.append({
        "attribute": name,
        "used": "no" if name in IGNORE else "yes",
        "description": DESCRIPTIONS[name],
        "distinct": int(series.nunique()),
        "example values": examples,
    })
profile = pd.DataFrame(rows)
display(profile)
print(f"Used columns ({len(used)}): {', '.join(used)}")



Step 1.5-1.6   Every attribute, what it means, and what the file actually contains


,attribute,used,description,distinct,example values
0,encounter_id,yes,Unique ID of one hospital stay,101766,"2278392, 149190, 64410"
1,patient_nbr,yes,ID of the patient. One patient can have several stays,71518,"8222157, 55629189, 86047875"
2,race,yes,"Caucasian, AfricanAmerican, Hispanic, Asian, or Other",6,"Caucasian, AfricanAmerican, Other"
3,gender,yes,"Female, Male, or Unknown/Invalid",3,"Female, Male, Unknown/Invalid"
4,age,yes,"Age band in 10-year groups, such as [70-80)",10,"[0-10), [10-20), [20-30)"
5,weight,yes,Weight band in pounds,10,"[75-100), [50-75), [0-25)"
6,admission_type_id,yes,"Code for the kind of admission (emergency, elective, and so on)",8,"6, 1, 2"
7,discharge_disposition_id,yes,Code for where the patient went at discharge,26,"25, 1, 3"
8,admission_source_id,yes,Code for where the patient was admitted from,17,"1, 7, 2"
9,time_in_hospital,yes,"Days in the hospital, from 1 to 14",14,"1, 3, 2"


Used columns (26): encounter_id, patient_nbr, race, gender, age, weight, admission_type_id, discharge_disposition_id, admission_source_id, time_in_hospital, payer_code, medical_specialty, num_lab_procedures, num_procedures, num_medications, number_outpatient, number_emergency, number_inpatient, diag_1, diag_2, diag_3, number_diagnoses, max_glu_serum, A1Cresult, diabetesMed, readmitted


In [4]:
banner("1.7", "Possible identifiers or keys")

print("A column can be the primary key of this file when it is filled in on every row")
print("and has a different value on every row.\n")
for name in df.columns:
    distinct = df[name].nunique()
    blank = df[name].isin({"?", ""}).sum()
    if distinct == len(df) and blank == 0:
        print(f"  KEY   {name}: {distinct:,} distinct values, one per row")

print("\nThese identifiers repeat, so they name another entity, not this row:")
for name in ("patient_nbr", "admission_type_id", "discharge_disposition_id", "admission_source_id"):
    print(f"  {name}: {df[name].nunique():,} distinct values across {len(df):,} stays")

codes = pd.concat([df["diag_1"], df["diag_2"], df["diag_3"]])
codes = codes[codes != "?"]
print(f"\ndiag_1, diag_2, and diag_3 are the same kind of value in three columns.")
print(f"Together they contain {codes.nunique():,} distinct ICD-9 codes.")
print("That repetition is the 1NF problem in Part 2. It is not a key of this file.")



Step 1.7   Possible identifiers or keys
A column can be the primary key of this file when it is filled in on every row
and has a different value on every row.

  KEY   encounter_id: 101,766 distinct values, one per row

These identifiers repeat, so they name another entity, not this row:
  patient_nbr: 71,518 distinct values across 101,766 stays
  admission_type_id: 8 distinct values across 101,766 stays
  discharge_disposition_id: 26 distinct values across 101,766 stays
  admission_source_id: 17 distinct values across 101,766 stays

diag_1, diag_2, and diag_3 are the same kind of value in three columns.
Together they contain 915 distinct ICD-9 codes.
That repetition is the 1NF problem in Part 2. It is not a key of this file.


In [5]:
banner("1.8", "Missing, unknown, or incomplete values")

print("The file does not use blank cells. It uses three different markers.\n")
print(f"{'column':24} {'marker':20} {'rows':>10} {'share':>8}")
any_marker = False
for name in used:
    series = df[name]
    for marker in ("?", "None", "Unknown/Invalid"):
        count = int((series == marker).sum())
        if count == 0:
            continue
        any_marker = True
        print(f"{name:24} {marker:20} {count:10,} {count / len(df):8.1%}")
if not any_marker:
    print("(none)")

print("\nHow each marker is stored in the database:")
print("  ?                unknown value        -> NULL")
print("  None             test was not run     -> NULL  (max_glu_serum, A1Cresult only)")
print("  Unknown/Invalid  gender not recorded  -> NULL")
print("Every other used column is filled in on every row.")



Step 1.8   Missing, unknown, or incomplete values
The file does not use blank cells. It uses three different markers.

column                   marker                     rows    share
race                     ?                         2,273     2.2%
gender                   Unknown/Invalid               3     0.0%


weight                   ?                        98,569    96.9%


payer_code               ?                        40,256    39.6%
medical_specialty        ?                        49,949    49.1%
diag_1                   ?                            21     0.0%
diag_2                   ?                           358     0.4%
diag_3                   ?                         1,423     1.4%
max_glu_serum            None                     96,420    94.7%
A1Cresult                None                     84,748    83.3%

How each marker is stored in the database:
  ?                unknown value        -> NULL
  None             test was not run     -> NULL  (max_glu_serum, A1Cresult only)
  Unknown/Invalid  gender not recorded  -> NULL
Every other used column is filled in on every row.


In [6]:
banner("1.9", "Which attributes stay the same for a patient")

def conflicting(column, drop):
    known = df.loc[~df[column].isin(drop), ["patient_nbr", column]]
    counts = known.groupby("patient_nbr")[column].nunique()
    return counts, int((counts > 1).sum())

race_counts, race_conflicts = conflicting("race", {"?"})
gender_counts, gender_conflicts = conflicting("gender", {"Unknown/Invalid"})
age_counts, age_changes = conflicting("age", set())

print(f"Patients with more than one recorded race:   {race_conflicts:,}")
print(f"Patients with more than one recorded gender: {gender_conflicts:,}")
print(f"Patients whose age band changes:    {age_changes:,}")
print()
print("race and gender are patient attributes. Where a patient has two values,")
print("the load keeps the value from the stay with the highest encounter_id.")
print("age changes often enough that it belongs to the stay, not the patient.")

sample_race = race_counts[race_counts > 1].head(1).index
if len(sample_race):
    print("\nOne patient with two races (question marks omitted):")
    display(
        df.loc[df["patient_nbr"].isin(sample_race) & ~df["race"].eq("?"),
               ["encounter_id", "patient_nbr", "race"]]
        .sort_values("encounter_id")
    )
sample_age = age_counts[age_counts > 1].head(1).index
print("One patient whose age band changes between stays:")
display(
    df.loc[df["patient_nbr"].isin(sample_age), ["encounter_id", "patient_nbr", "age"]]
    .assign(_id=lambda frame: frame["encounter_id"].astype("int64"))
    .sort_values("_id")
    .drop(columns="_id")
)



Step 1.9   Which attributes stay the same for a patient


Patients with more than one recorded race:   123
Patients with more than one recorded gender: 3
Patients whose age band changes:    1,574

race and gender are patient attributes. Where a patient has two values,
the load keeps the value from the stay with the highest encounter_id.
age changes often enough that it belongs to the stay, not the patient.

One patient with two races (question marks omitted):


,encounter_id,patient_nbr,race
51411,153403926,100322946,Hispanic
72127,210542046,100322946,Other
75032,224410320,100322946,Other


One patient whose age band changes between stays:


,encounter_id,patient_nbr,age
37559,116546898,100018278,[60-70)
79795,245361768,100018278,[70-80)


## Part 2. Normalize the dataset

The assignment requires the original relation, then 1NF, then 2NF. At each stage the report needs six things: structure, key, functional dependencies, the problem, the change, and why the result is in that form.

A functional dependency A → B holds when each value of A pairs with only one value of B. The check below is the largest number of distinct B values inside one A group. The dependency holds when that number is 1.


In [7]:
banner("2.1", "Original relation")

def dependency_width(frame, determinant, dependent, drop=()):
    subset = frame.loc[~frame[dependent].isin(drop)] if drop else frame
    if subset.empty:
        return 0
    return int(subset.groupby(determinant)[dependent].nunique().max())

print("1. Structure: DIABETIC_DATA(" + ", ".join(used) + ")")
print("   one row per stay. Rows:", f"{len(df):,}")
print("2. Key: encounter_id")
print(f"   distinct encounter_id = {df['encounter_id'].nunique():,}, equal to the row count.")
print("3. Functional dependencies that hold because the key is unique:")
print("   encounter_id -> every other column")
for column, drop in (("race", {"?"}), ("gender", {"Unknown/Invalid"}), ("age", ())):
    width = dependency_width(df, "patient_nbr", column, drop)
    status = "holds" if width <= 1 else "does not hold"
    print(f"   patient_nbr -> {column:8}  max distinct values per patient = {width}  ({status})")
print("4. Problem: diag_1, diag_2, and diag_3 are a repeating group.")
print("   One stay stores up to three ICD-9 codes in three columns of the same kind.")
sample = df.loc[(df["diag_1"] != "?") & (df["diag_2"] != "?") & (df["diag_3"] != "?")].iloc[0]
print("   One stay, three codes:")
display(sample[["encounter_id", "diag_1", "diag_2", "diag_3", "time_in_hospital"]].to_frame().T)
print("5. Change: one row per diagnosis, with a new column diag_position (1, 2, or 3).")
print("6. Form: not 1NF, because the diagnosis columns repeat.")



Step 2.1   Original relation
1. Structure: DIABETIC_DATA(encounter_id, patient_nbr, race, gender, age, weight, admission_type_id, discharge_disposition_id, admission_source_id, time_in_hospital, payer_code, medical_specialty, num_lab_procedures, num_procedures, num_medications, number_outpatient, number_emergency, number_inpatient, diag_1, diag_2, diag_3, number_diagnoses, max_glu_serum, A1Cresult, diabetesMed, readmitted)
   one row per stay. Rows: 101,766
2. Key: encounter_id
   distinct encounter_id = 101,766, equal to the row count.
3. Functional dependencies that hold because the key is unique:
   encounter_id -> every other column
   patient_nbr -> race      max distinct values per patient = 3  (does not hold)


   patient_nbr -> gender    max distinct values per patient = 2  (does not hold)
   patient_nbr -> age       max distinct values per patient = 2  (does not hold)
4. Problem: diag_1, diag_2, and diag_3 are a repeating group.
   One stay stores up to three ICD-9 codes in three columns of the same kind.


   One stay, three codes:


,encounter_id,diag_1,diag_2,diag_3,time_in_hospital
1,149190,276,250.01,255,3


5. Change: one row per diagnosis, with a new column diag_position (1, 2, or 3).
6. Form: not 1NF, because the diagnosis columns repeat.


In [8]:
banner("2.2", "First normal form")

pieces = []
for position, column in ((1, "diag_1"), (2, "diag_2"), (3, "diag_3")):
    present = df.loc[df[column] != "?", ["encounter_id", column]].copy()
    present.columns = ["encounter_id", "icd9_code"]
    present["diag_position"] = position
    pieces.append(present)
diagnosis_rows = pd.concat(pieces, ignore_index=True)

pair_repeats = (
    ((df["diag_1"] == df["diag_2"]) & df["diag_1"].ne("?"))
    | ((df["diag_1"] == df["diag_3"]) & df["diag_1"].ne("?"))
    | ((df["diag_2"] == df["diag_3"]) & df["diag_2"].ne("?"))
)
duplicate_pairs = int(diagnosis_rows.duplicated(["encounter_id", "icd9_code"]).sum())
stays_with_a_diag = diagnosis_rows["encounter_id"].nunique()
stays_with_none = len(df) - stays_with_a_diag

print("1. Structure: one row per recorded diagnosis")
print("   ENCOUNTER_1NF(encounter_id, diag_position, icd9_code, <every stay column>)")
print(f"   diagnosis rows: {len(diagnosis_rows):,}")
print(f"   stays with no diagnosis code at all: {stays_with_none:,}")
print("   Those stays are kept later, on ENCOUNTER. They simply have no diagnosis rows.")
print("2. Key: (encounter_id, diag_position)")
key_dups = int(diagnosis_rows.duplicated(["encounter_id", "diag_position"]).sum())
print(f"   duplicate (encounter_id, diag_position) pairs: {key_dups}")
print(f"   Stays that list the same ICD-9 code twice: {int(pair_repeats.sum()):,}")
print(f"   Extra (encounter_id, icd9_code) copies: {duplicate_pairs:,}")
print("   (encounter_id, icd9_code) is not a key, because those pairs are not unique.")
print("3. Functional dependencies:")
print("   (encounter_id, diag_position) -> icd9_code")
print("   encounter_id -> every stay column  (this is the 2NF problem)")
print("4. Problem: stay facts are repeated once per diagnosis of that stay.")
shown = sample["encounter_id"]
long = diagnosis_rows.loc[diagnosis_rows["encounter_id"] == shown].merge(
    df.loc[df["encounter_id"] == shown, ["encounter_id", "time_in_hospital"]],
    on="encounter_id",
)
print(f"   Stay {shown} in 1NF. time_in_hospital is copied onto every diagnosis row:")
display(long[["encounter_id", "diag_position", "icd9_code", "time_in_hospital"]])
print("5. Change: move stay columns onto ENCOUNTER, whose key is encounter_id only.")
print("   Leave ENCOUNTER_DIAGNOSIS(encounter_id, diag_position, icd9_code).")
print("6. Form: 1NF, because every cell holds one value. Not yet 2NF.")



Step 2.2   First normal form


1. Structure: one row per recorded diagnosis
   ENCOUNTER_1NF(encounter_id, diag_position, icd9_code, <every stay column>)
   diagnosis rows: 303,496
   stays with no diagnosis code at all: 1
   Those stays are kept later, on ENCOUNTER. They simply have no diagnosis rows.
2. Key: (encounter_id, diag_position)
   duplicate (encounter_id, diag_position) pairs: 0
   Stays that list the same ICD-9 code twice: 5,492
   Extra (encounter_id, icd9_code) copies: 5,591
   (encounter_id, icd9_code) is not a key, because those pairs are not unique.
3. Functional dependencies:
   (encounter_id, diag_position) -> icd9_code
   encounter_id -> every stay column  (this is the 2NF problem)
4. Problem: stay facts are repeated once per diagnosis of that stay.
   Stay 149190 in 1NF. time_in_hospital is copied onto every diagnosis row:


,encounter_id,diag_position,icd9_code,time_in_hospital
0,149190,1,276,3
1,149190,2,250.01,3
2,149190,3,255,3


5. Change: move stay columns onto ENCOUNTER, whose key is encounter_id only.
   Leave ENCOUNTER_DIAGNOSIS(encounter_id, diag_position, icd9_code).
6. Form: 1NF, because every cell holds one value. Not yet 2NF.


In [9]:
banner("2.3", "Partial dependencies")

print("The 1NF key has two columns. A non-key column that depends on only one of them")
print("is a partial dependency, which 2NF does not allow.\n")
per_encounter = diagnosis_rows.groupby("encounter_id")["icd9_code"].nunique()
per_full_key = diagnosis_rows.groupby(["encounter_id", "diag_position"])["icd9_code"].nunique()
print(f"icd9_code  distinct values per encounter_id:              max {int(per_encounter.max())}")
print(f"icd9_code  distinct values per (encounter_id, position):  max {int(per_full_key.max())}")
print("icd9_code needs the whole key.\n")

print("Every original stay column has exactly 1 distinct value per encounter_id,")
print("because encounter_id was already unique in the CSV. After the 1NF split,")
print("that column is repeated and still depends only on encounter_id.")
for column in ("time_in_hospital", "age", "readmitted", "patient_nbr", "admission_type_id"):
    width = int(df.groupby("encounter_id")[column].nunique().max())
    print(f"  encounter_id -> {column:22} max distinct = {width}   partial dependency")

print("\nENCOUNTER gets those columns. Its key is the single column encounter_id,")
print("so nothing inside it can depend on part of the key. A one-column key is in 2NF.")



Step 2.3   Partial dependencies
The 1NF key has two columns. A non-key column that depends on only one of them
is a partial dependency, which 2NF does not allow.



icd9_code  distinct values per encounter_id:              max 3
icd9_code  distinct values per (encounter_id, position):  max 1
icd9_code needs the whole key.

Every original stay column has exactly 1 distinct value per encounter_id,
because encounter_id was already unique in the CSV. After the 1NF split,
that column is repeated and still depends only on encounter_id.
  encounter_id -> time_in_hospital       max distinct = 1   partial dependency


  encounter_id -> age                    max distinct = 1   partial dependency


  encounter_id -> readmitted             max distinct = 1   partial dependency
  encounter_id -> patient_nbr            max distinct = 1   partial dependency
  encounter_id -> admission_type_id      max distinct = 1   partial dependency

ENCOUNTER gets those columns. Its key is the single column encounter_id,
so nothing inside it can depend on part of the key. A one-column key is in 2NF.


In [10]:
banner("2.4", "Second normal form: the seven tables")

numbers = {name: pd.to_numeric(df[name], errors="raise").astype("int64") for name in INT_COLUMNS}
icd9 = pd.concat([df["diag_1"], df["diag_2"], df["diag_3"]])
icd9 = icd9[icd9 != "?"].drop_duplicates()

print("1. Structure: seven tables. Lookup descriptions come from IDS_mapping.csv.")
print("2. Keys: one column, except ENCOUNTER_DIAGNOSIS, whose key is the pair below.")
print("3. Functional dependencies: each non-key column depends on the whole key of its table.")
print("   On ENCOUNTER_DIAGNOSIS the only non-key column is icd9_code, and Step 2.3")
print("   showed that it depends on both key columns.")
print("4. Problem: none left. No column depends on only part of a key.")
print("   Race and gender stay on ENCOUNTER. patient_nbr does not determine them.")
print(f"   {race_conflicts:,} patients have more than one race, and {gender_conflicts:,} have more than one gender.")
print("5. Change: none. 2NF is the form this assignment asks for.")
print("6. Form: 2NF. No partial dependencies remain.\n")

schema = pd.DataFrame([
    ["PATIENT", "patient_nbr", "patient_nbr", ""],
    ["ADMISSION_TYPE", "admission_type_id, description", "admission_type_id", ""],
    ["DISCHARGE_DISPOSITION", "discharge_disposition_id, description", "discharge_disposition_id", ""],
    ["ADMISSION_SOURCE", "admission_source_id, description", "admission_source_id", ""],
    ["DIAGNOSIS", "icd9_code", "icd9_code", ""],
    ["ENCOUNTER",
     "encounter_id, patient_nbr, race, gender, admission_type_id, discharge_disposition_id, admission_source_id, age, weight, time_in_hospital, payer_code, medical_specialty, num_lab_procedures, num_procedures, num_medications, number_outpatient, number_emergency, number_inpatient, number_diagnoses, max_glu_serum, A1Cresult, diabetesMed, readmitted",
     "encounter_id",
     "patient_nbr -> PATIENT; admission_type_id -> ADMISSION_TYPE; discharge_disposition_id -> DISCHARGE_DISPOSITION; admission_source_id -> ADMISSION_SOURCE"],
    ["ENCOUNTER_DIAGNOSIS", "encounter_id, diag_position, icd9_code",
     "(encounter_id, diag_position)",
     "encounter_id -> ENCOUNTER; icd9_code -> DIAGNOSIS"],
], columns=["table", "attributes", "primary key", "foreign keys"])
display(schema)

print("Row counts these tables will have after the load:")
print(f"  PATIENT                 {df['patient_nbr'].nunique():,}")
print(f"  ENCOUNTER               {len(df):,}")
print(f"  ENCOUNTER_DIAGNOSIS     {len(diagnosis_rows):,}")
print(f"  DIAGNOSIS               {icd9.nunique():,}")
for column, table in (
    ("admission_type_id", "ADMISSION_TYPE"),
    ("discharge_disposition_id", "DISCHARGE_DISPOSITION"),
    ("admission_source_id", "ADMISSION_SOURCE"),
):
    present = set(df[column].unique())
    known = set(ID_MAPS[column])
    print(f"  {table:24} {len(present):,}")
    missing_from_map = sorted(present - known, key=int)
    if missing_from_map:
        print(f"    ids with no description in IDS_mapping.csv: {missing_from_map}")



Step 2.4   Second normal form: the seven tables


1. Structure: seven tables. Lookup descriptions come from IDS_mapping.csv.
2. Keys: one column, except ENCOUNTER_DIAGNOSIS, whose key is the pair below.
3. Functional dependencies: each non-key column depends on the whole key of its table.
   On ENCOUNTER_DIAGNOSIS the only non-key column is icd9_code, and Step 2.3
   showed that it depends on both key columns.
4. Problem: none left. Each fact is stored once.
   race and gender move to PATIENT so they are not repeated on every stay.
   123 patients have more than one race, and 3 have more than one gender.
   The load keeps the latest stay. Step 7.4 counts the stays that then differ from the CSV.
5. Change: none. This is the final design.
6. Form: 2NF. No partial dependencies remain.



,table,attributes,primary key,foreign keys
0,PATIENT,"patient_nbr, race, gender",patient_nbr,
1,ADMISSION_TYPE,"admission_type_id, description",admission_type_id,
2,DISCHARGE_DISPOSITION,"discharge_disposition_id, description",discharge_disposition_id,
3,ADMISSION_SOURCE,"admission_source_id, description",admission_source_id,
4,DIAGNOSIS,icd9_code,icd9_code,
5,ENCOUNTER,"encounter_id, patient_nbr, admission_type_id, discharge_disposition_id, admission_so...",encounter_id,patient_nbr -> PATIENT; admission_type_id -> ADMISSION_TYPE; discharge_disposition_i...
6,ENCOUNTER_DIAGNOSIS,"encounter_id, diag_position, icd9_code","(encounter_id, diag_position)",encounter_id -> ENCOUNTER; icd9_code -> DIAGNOSIS


Row counts these tables will have after the load:
  PATIENT                 71,518
  ENCOUNTER               101,766
  ENCOUNTER_DIAGNOSIS     303,496
  DIAGNOSIS               915
  ADMISSION_TYPE           8
  DISCHARGE_DISPOSITION    26
  ADMISSION_SOURCE         17


## Part 3. Construct an ER diagram

Cardinality is computed from the file: the maximum number of rows on each side of a relationship. Participation is the constraint we record on the diagram. Where this extract happens to use every lookup code, the diagram can still say the lookup side is optional, because a code is allowed to exist before any stay uses it.


In [11]:
banner("3.1-3.3", "Entities, cardinality, and participation")

enc_per_patient = df.groupby("patient_nbr").size()
patients_per_enc = df.groupby("encounter_id")["patient_nbr"].nunique()
diag_per_enc_all = diagnosis_rows.groupby("encounter_id").size()
enc_per_code = diagnosis_rows.groupby("icd9_code").size()

print(f"Patients per stay:     min {int(patients_per_enc.min())}  max {int(patients_per_enc.max())}")
print(f"Stays per patient:     min {int(enc_per_patient.min())}  max {int(enc_per_patient.max())}")
print(f"Diagnoses per stay:    min {int(diag_per_enc_all.min())}  max {int(diag_per_enc_all.max())}  (among stays that have one)")
print(f"Stays with no diagnosis:{stays_with_none:,}")
print(f"Stays per ICD-9 code:  min {int(enc_per_code.min())}  max {int(enc_per_code.max())}")
print()

for column, label in (
    ("admission_type_id", "admission type"),
    ("discharge_disposition_id", "discharge disposition"),
    ("admission_source_id", "admission source"),
):
    per_code = df.groupby(column).size()
    per_stay = df.groupby("encounter_id")[column].nunique()
    print(f"{label}: codes per stay min {int(per_stay.min())} max {int(per_stay.max())}; "
          f"stays per stored code min {int(per_code.min())} max {int(per_code.max())}")

print("\nReading of those counts:")
print("  A stay has one patient, and a patient has many stays            -> 1:N, total on both sides")
print("  A stay has one admission type, and a type covers many stays     -> 1:N")
print("  Same pattern for discharge disposition and admission source     -> 1:N")
print("  A stay has up to 3 diagnoses, and a code appears in many stays  -> M:N")
print("  The M:N relationship is drawn as two 1:N relationships through ENCOUNTER_DIAGNOSIS.")
print("  diag_position is an attribute of that relationship.")

# parent_optional: the one-side may have zero children.
# child_optional: a child may lack a parent. None of these relationships allow that.
def crow(parent_optional, child_optional=False):
    at_parent = "|o" if child_optional else "||"
    at_child = "o{" if parent_optional else "|{"
    return f"{at_parent}--{at_child}"

assert crow(False) == "||--|{"
assert crow(True) == "||--o{"

RELATIONSHIPS = [
    {"name": "has", "one": "PATIENT", "many": "ENCOUNTER",
     "optional_one": False, "fk": "ENCOUNTER.patient_nbr -> PATIENT.patient_nbr",
     "why": "a stay has one patient; every patient in this file has at least one stay"},
    {"name": "classifies", "one": "ADMISSION_TYPE", "many": "ENCOUNTER",
     "optional_one": False, "fk": "ENCOUNTER.admission_type_id -> ADMISSION_TYPE.admission_type_id",
     "why": "a stay has one type; every type in this file is used"},
    {"name": "ends with", "one": "DISCHARGE_DISPOSITION", "many": "ENCOUNTER",
     "optional_one": False, "fk": "ENCOUNTER.discharge_disposition_id -> DISCHARGE_DISPOSITION.discharge_disposition_id",
     "why": "a stay has one disposition; every disposition in this file is used"},
    {"name": "admitted from", "one": "ADMISSION_SOURCE", "many": "ENCOUNTER",
     "optional_one": False, "fk": "ENCOUNTER.admission_source_id -> ADMISSION_SOURCE.admission_source_id",
     "why": "a stay has one source; every source in this file is used"},
    {"name": "diagnosed with", "one": "ENCOUNTER", "many": "ENCOUNTER_DIAGNOSIS",
     "optional_one": True, "fk": "ENCOUNTER_DIAGNOSIS.encounter_id -> ENCOUNTER.encounter_id",
     "why": f"a stay has zero to three diagnosis rows; {stays_with_none:,} stays have zero"},
    {"name": "appears in", "one": "DIAGNOSIS", "many": "ENCOUNTER_DIAGNOSIS",
     "optional_one": False, "fk": "ENCOUNTER_DIAGNOSIS.icd9_code -> DIAGNOSIS.icd9_code",
     "why": "every stored code appears in at least one stay, and every diagnosis row has one code"},
]



Step 3.1-3.3   Entities, cardinality, and participation


Patients per stay:     min 1  max 1
Stays per patient:     min 1  max 40
Diagnoses per stay:    min 1  max 3  (among stays that have one)
Stays with no diagnosis:1
Stays per ICD-9 code:  min 1  max 18101

admission type: codes per stay min 1 max 1; stays per stored code min 10 max 53990


discharge disposition: codes per stay min 1 max 1; stays per stored code min 2 max 60234


admission source: codes per stay min 1 max 1; stays per stored code min 1 max 57494

Reading of those counts:
  A stay has one patient, and a patient has many stays            -> 1:N, total on both sides
  A stay has one admission type, and a type covers many stays     -> 1:N
  Same pattern for discharge disposition and admission source     -> 1:N
  A stay has up to 3 diagnoses, and a code appears in many stays  -> M:N
  The M:N relationship is drawn as two 1:N relationships through ENCOUNTER_DIAGNOSIS.
  diag_position is an attribute of that relationship.


In [12]:
banner("3.4", "ER diagram")

print("Crow's foot, the same notation as the report:")
print("  ||    exactly one")
print("  |{    one or many (bar beside the crow's foot: total participation)")
print("  o{    zero or many (circle beside the crow's foot: partial participation)")
print()
lines = ["erDiagram"]
for rel in RELATIONSHIPS:
    symbol = crow(rel["optional_one"])
    label = f'{rel["name"]} (1:N)'
    lines.append(f'    {rel["one"]} {symbol} {rel["many"]} : "{label}"')
mermaid = "\n".join(lines)
print(mermaid)
print()
for rel in RELATIONSHIPS:
    side = "partial" if rel["optional_one"] else "total"
    print(f'{rel["one"]:24} {rel["name"]:16} {rel["many"]:24} 1:N  one-side {side}')
    print(f'    {rel["why"]}')
print("\nThere is no 1:1 relationship.")
print("ENCOUNTER diagnosed-with DIAGNOSIS is many-to-many. The two lines through")
print("ENCOUNTER_DIAGNOSIS are that many-to-many relationship, plus diag_position.")
display(Markdown("```mermaid\n" + mermaid + "\n```"))



Step 3.4   ER diagram
Crow's foot, the same notation as the report:
  ||    exactly one
  |{    one or many (bar beside the crow's foot: total participation)
  o{    zero or many (circle beside the crow's foot: partial participation)

erDiagram
    PATIENT ||--|{ ENCOUNTER : "has (1:N)"
    ADMISSION_TYPE ||--o{ ENCOUNTER : "classifies (1:N)"
    DISCHARGE_DISPOSITION ||--o{ ENCOUNTER : "ends with (1:N)"
    ADMISSION_SOURCE ||--o{ ENCOUNTER : "admitted from (1:N)"
    ENCOUNTER ||--o{ ENCOUNTER_DIAGNOSIS : "records (1:N)"
    DIAGNOSIS ||--|{ ENCOUNTER_DIAGNOSIS : "appears in (1:N)"

PATIENT                  has              ENCOUNTER                1:N  one-side total
    a stay has one patient; every patient in this file has at least one stay
ADMISSION_TYPE           classifies       ENCOUNTER                1:N  one-side partial
    a stay has one type; a type is allowed to have no stays
DISCHARGE_DISPOSITION    ends with        ENCOUNTER                1:N  one-side partial
    a

```mermaid
erDiagram
    PATIENT ||--|{ ENCOUNTER : "has (1:N)"
    ADMISSION_TYPE ||--o{ ENCOUNTER : "classifies (1:N)"
    DISCHARGE_DISPOSITION ||--o{ ENCOUNTER : "ends with (1:N)"
    ADMISSION_SOURCE ||--o{ ENCOUNTER : "admitted from (1:N)"
    ENCOUNTER ||--o{ ENCOUNTER_DIAGNOSIS : "records (1:N)"
    DIAGNOSIS ||--|{ ENCOUNTER_DIAGNOSIS : "appears in (1:N)"
```

## Part 4. Convert the ER diagram into relational tables

Three rules. Each entity becomes a table. Each 1:N relationship becomes a foreign key on the many side. Each M:N relationship becomes its own table; that table is already ENCOUNTER_DIAGNOSIS.


In [13]:
banner("4.1", "Each entity becomes a table")
for table in (
    "PATIENT", "ADMISSION_TYPE", "DISCHARGE_DISPOSITION", "ADMISSION_SOURCE",
    "DIAGNOSIS", "ENCOUNTER", "ENCOUNTER_DIAGNOSIS",
):
    print(f"  {table}")

banner("4.2", "Each 1:N relationship becomes a foreign key on the many side")
for rel in RELATIONSHIPS:
    print(f'  {rel["name"]:16} {rel["fk"]}')

banner("4.3", "The many-to-many relationship")
print("  ENCOUNTER diagnosed-with DIAGNOSIS")
print("  -> ENCOUNTER_DIAGNOSIS(encounter_id, diag_position, icd9_code)")
print("     foreign key to ENCOUNTER, foreign key to DIAGNOSIS,")
print("     and diag_position, which belongs to the relationship.")

banner("4.4", "Creation order")
print("A table is created after every table its foreign keys point to.")
print("  1. PATIENT, ADMISSION_TYPE, DISCHARGE_DISPOSITION, ADMISSION_SOURCE, DIAGNOSIS")
print("  2. ENCOUNTER")
print("  3. ENCOUNTER_DIAGNOSIS")
print("Creating ENCOUNTER before PATIENT fails: the foreign key would name a missing table.")



Step 4.1   Each entity becomes a table
  PATIENT
  ADMISSION_TYPE
  DISCHARGE_DISPOSITION
  ADMISSION_SOURCE
  DIAGNOSIS
  ENCOUNTER
  ENCOUNTER_DIAGNOSIS

Step 4.2   Each 1:N relationship becomes a foreign key on the many side
  has              ENCOUNTER.patient_nbr -> PATIENT.patient_nbr
  classifies       ENCOUNTER.admission_type_id -> ADMISSION_TYPE.admission_type_id
  ends with        ENCOUNTER.discharge_disposition_id -> DISCHARGE_DISPOSITION.discharge_disposition_id
  admitted from    ENCOUNTER.admission_source_id -> ADMISSION_SOURCE.admission_source_id
  records          ENCOUNTER_DIAGNOSIS.encounter_id -> ENCOUNTER.encounter_id
  appears in       ENCOUNTER_DIAGNOSIS.icd9_code -> DIAGNOSIS.icd9_code

Step 4.3   The many-to-many relationship
  ENCOUNTER diagnosed-with DIAGNOSIS
  -> ENCOUNTER_DIAGNOSIS(encounter_id, diag_position, icd9_code)
     foreign key to ENCOUNTER, foreign key to DIAGNOSIS,
     and diag_position, which belongs to the relationship.

Step 4.4   Creation 

## Part 5. Implement the database

Types, `NOT NULL`, and `CHECK` are decided from the values in the file. MySQL `INT` only goes to 2,147,483,647; a wider id becomes `BIGINT`. A column that contains `?`, `None`, or `Unknown/Invalid` is nullable, and those markers become NULL at load time.

The statements run here in SQLite. Step 7.6 writes the same statements as MySQL.


In [14]:
banner("5.1", "Choose a type for every stored column")

TEXT_RULES = {
    "race": {"?"},
    "gender": {"Unknown/Invalid"},
    "age": set(),
    "weight": {"?"},
    "payer_code": {"?"},
    "medical_specialty": {"?"},
    "max_glu_serum": {"None"},
    "A1Cresult": {"None"},
    "diabetesMed": set(),
    "readmitted": set(),
}
ENCOUNTER_INT = [
    "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
    "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses",
]
ENCOUNTER_FK = [
    ("patient_nbr", "patient", "patient_nbr"),
    ("admission_type_id", "admission_type", "admission_type_id"),
    ("discharge_disposition_id", "discharge_disposition", "discharge_disposition_id"),
    ("admission_source_id", "admission_source", "admission_source_id"),
]

decisions = []
for name, tokens in TEXT_RULES.items():
    sql_type, observed = mysql_varchar(df[name], tokens)
    nullable = bool(df[name].isin(tokens).any()) if tokens else False
    decisions.append((name, sql_type, observed, nullable))
    print(f"  {name:24} longest real value {observed:3} -> {sql_type:14} "
          f"{'NULL allowed' if nullable else 'NOT NULL'}")

for name in ["encounter_id", "patient_nbr", *ENCOUNTER_INT, "admission_type_id",
             "discharge_disposition_id", "admission_source_id"]:
    sql_type = mysql_int(df[name])
    observed = int(df[name].astype("int64").max())
    print(f"  {name:24} max {observed:>12,} -> {sql_type}")

tih = df["time_in_hospital"].astype(int)
meds = df["num_medications"].astype(int)
print(f"\n  time_in_hospital ranges from {int(tih.min())} to {int(tih.max())}")
print("  CHECK (time_in_hospital BETWEEN 1 AND 14) accepts every row.")
print(f"  num_medications ranges from {int(meds.min())} to {int(meds.max())}")
print("  CHECK (diag_position IN (1, 2, 3)) matches the 1NF positions.")

icd_type, icd_len = mysql_varchar(icd9)
desc_values = pd.Series([v for table in ID_MAPS.values() for v in table.values() if v])
desc_type, desc_len = mysql_varchar(desc_values)
print(f"  icd9_code longest {icd_len} -> {icd_type}")
print(f"  description longest {desc_len} -> {desc_type}")


def column_sql(name, sql_type, *, null=True, check=None):
    pieces = [name, sql_type]
    if not null:
        pieces.append("NOT NULL")
    if check:
        pieces.append(f"CHECK ({check})")
    return " ".join(pieces)


def create_statement(table, columns, extras):
    body = [f"    {item}" for item in [*columns, *extras]]
    return f"CREATE TABLE {table} (\n" + ",\n".join(body) + "\n);"


type_of = {name: sql_type for name, sql_type, _, _ in decisions}
null_of = {name: nullable for name, _, _, nullable in decisions}

patient_cols = [
    f"patient_nbr {mysql_int(df['patient_nbr'])} PRIMARY KEY",
]
code_tables = []
for column, table in (
    ("admission_type_id", "admission_type"),
    ("discharge_disposition_id", "discharge_disposition"),
    ("admission_source_id", "admission_source"),
):
    code_tables.append(create_statement(
        table,
        [f"{column} {mysql_int(df[column])} PRIMARY KEY",
         column_sql("description", desc_type, null=True)],
        [],
    ))

encounter_cols = [f"encounter_id {mysql_int(df['encounter_id'])} PRIMARY KEY"]
for column, parent, parent_col in ENCOUNTER_FK:
    encounter_cols.append(
        f"{column} {mysql_int(df[column])} NOT NULL"
    )
for name in ("race", "gender", "age", "weight", "payer_code", "medical_specialty"):
    encounter_cols.append(column_sql(name, type_of[name], null=null_of[name]))
encounter_cols.append(
    column_sql("time_in_hospital", "INT", null=False,
               check="time_in_hospital BETWEEN 1 AND 14")
)
for name in ENCOUNTER_INT[1:]:
    encounter_cols.append(column_sql(name, "INT", null=False))
for name in ("max_glu_serum", "A1Cresult", "diabetesMed", "readmitted"):
    encounter_cols.append(column_sql(name, type_of[name], null=null_of[name]))

encounter_fks = [
    f"CONSTRAINT fk_enc_{column} FOREIGN KEY ({column}) REFERENCES {parent} ({parent_col})"
    for column, parent, parent_col in ENCOUNTER_FK
]
bridge_cols = [
    f"encounter_id {mysql_int(df['encounter_id'])} NOT NULL",
    "diag_position INT NOT NULL CHECK (diag_position IN (1, 2, 3))",
    column_sql("icd9_code", icd_type, null=False),
]
bridge_extras = [
    "PRIMARY KEY (encounter_id, diag_position)",
    "FOREIGN KEY (encounter_id) REFERENCES encounter (encounter_id)",
    "FOREIGN KEY (icd9_code) REFERENCES diagnosis (icd9_code)",
]

CREATE_BODY = "\n\n".join([
    create_statement("patient", patient_cols, []),
    *code_tables,
    create_statement("diagnosis", [f"icd9_code {icd_type} PRIMARY KEY"], []),
    create_statement("encounter", encounter_cols, encounter_fks),
    create_statement("encounter_diagnosis", bridge_cols, bridge_extras),
])

banner("5.2", "Create the tables")
print(CREATE_BODY)

if "con" in globals():
    try:
        con.close()
    except Exception:
        pass
con = sqlite3.connect(DB_PATH)
con.isolation_level = None  # we commit each statement ourselves
con.execute("PRAGMA foreign_keys = OFF")
for table in (
    "encounter_diagnosis", "encounter", "patient", "diagnosis",
    "admission_type", "discharge_disposition", "admission_source", "staging",
):
    con.execute(f"DROP TABLE IF EXISTS {table}")
con.execute("PRAGMA foreign_keys = ON")
fk_on = con.execute("PRAGMA foreign_keys").fetchone()[0]
if fk_on != 1:
    raise RuntimeError("SQLite foreign keys are off. Later checks would be meaningless.")

for statement in CREATE_BODY.split(";"):
    if statement.strip():
        con.execute(statement)
created = [row[0] for row in con.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
)]
print("\nSQLite tables:", ", ".join(created))
print(f"Foreign keys enforced: {bool(fk_on)}")



Step 5.1   Choose a type for every stored column
  race                     longest real value  15 -> VARCHAR(20)    NULL allowed
  gender                   longest real value   6 -> VARCHAR(10)    NULL allowed
  age                      longest real value   8 -> VARCHAR(10)    NOT NULL
  weight                   longest real value   9 -> VARCHAR(10)    NULL allowed
  payer_code               longest real value   2 -> VARCHAR(10)    NULL allowed
  medical_specialty        longest real value  36 -> VARCHAR(40)    NULL allowed
  max_glu_serum            longest real value   4 -> VARCHAR(10)    NULL allowed
  A1Cresult                longest real value   4 -> VARCHAR(10)    NULL allowed
  diabetesMed              longest real value   3 -> VARCHAR(10)    NOT NULL
  readmitted               longest real value   3 -> VARCHAR(10)    NOT NULL
  encounter_id             max  443,867,222 -> INT
  patient_nbr              max  189,502,619 -> INT
  time_in_hospital         max           14 -> INT

  number_inpatient         max           21 -> INT
  number_diagnoses         max           16 -> INT
  admission_type_id        max            8 -> INT
  discharge_disposition_id max           28 -> INT
  admission_source_id      max           25 -> INT

  time_in_hospital ranges from 1 to 14
  CHECK (time_in_hospital BETWEEN 1 AND 14) accepts every row.
  num_medications ranges from 1 to 81
  CHECK (diag_position IN (1, 2, 3)) matches the 1NF positions.
  icd9_code longest 6 -> VARCHAR(10)
  description longest 105 -> VARCHAR(160)

Step 5.2   Create the tables
CREATE TABLE patient (
    patient_nbr INT PRIMARY KEY,
    race VARCHAR(20),
    gender VARCHAR(10)
);

CREATE TABLE admission_type (
    admission_type_id INT PRIMARY KEY,
    description VARCHAR(160)
);

CREATE TABLE discharge_disposition (
    discharge_disposition_id INT PRIMARY KEY,
    description VARCHAR(160)
);

CREATE TABLE admission_source (
    admission_source_id INT PRIMARY KEY,
    description VARCHAR(160)
);

CR


SQLite tables: admission_source, admission_type, diagnosis, discharge_disposition, encounter, encounter_diagnosis, patient
Foreign keys enforced: True


## Part 6. Import the CSV

The path is CSV file, then a staging table with one column per CSV column, then the seven tables. Parents are loaded before `encounter`, and `encounter` before `encounter_diagnosis`. Re-run from Step 6.1 if you run this part again: 6.1 clears the seven tables first.


In [15]:
banner("6.1", "Load the CSV into staging")

for table in (
    "encounter_diagnosis", "encounter", "patient", "diagnosis",
    "admission_type", "discharge_disposition", "admission_source",
):
    con.execute(f"DELETE FROM {table}")
con.execute("DROP TABLE IF EXISTS staging")

staging_columns = []
for name in df.columns:
    kind = "INTEGER" if name in INT_COLUMNS else "TEXT"
    staging_columns.append(f"{quote_sqlite(name)} {kind}")
con.execute("CREATE TABLE staging (" + ", ".join(staging_columns) + ")")

insert_sql = (
    "INSERT INTO staging (" + ", ".join(quote_sqlite(name) for name in df.columns) + ") "
    "VALUES (" + ", ".join(["?"] * len(df.columns)) + ")"
)
int_positions = [df.columns.get_loc(name) for name in INT_COLUMNS]
batch = []
for record in df.itertuples(index=False, name=None):
    row = list(record)
    for position in int_positions:
        row[position] = int(row[position])
    batch.append(tuple(row))
    if len(batch) == 5000:
        con.executemany(insert_sql, batch)
        batch.clear()
if batch:
    con.executemany(insert_sql, batch)

staged = con.execute("SELECT COUNT(*) FROM staging").fetchone()[0]
print(f"Staging rows: {staged:,}   (CSV rows: {len(df):,})")
if staged != len(df):
    raise RuntimeError("Staging did not receive every CSV row.")



Step 6.1   Load the CSV into staging


Staging rows: 101,766   (CSV rows: 101,766)


In [16]:
banner("6.2", "Parent tables: patients, lookup codes, diagnoses")

LOAD_SQL = []

code_inserts = []
for column, table in (
    ("admission_type_id", "admission_type"),
    ("discharge_disposition_id", "discharge_disposition"),
    ("admission_source_id", "admission_source"),
):
    mapping = ID_MAPS[column]
    ids = sorted(df[column].unique(), key=int)
    values = ",\n".join(
        f"    ({int(code)}, {sql_literal(mapping.get(code))})" for code in ids
    )
    statement = (
        f"INSERT INTO {table} ({column}, description) VALUES\n{values}"
    )
    code_inserts.append(statement)
    con.execute(statement)
    print(f"  {table}: {con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]:,} rows")

patient_sql = """
INSERT INTO patient (patient_nbr)
SELECT DISTINCT patient_nbr
FROM staging
""".strip()
con.execute(patient_sql)
print(f"  patient: {con.execute('SELECT COUNT(*) FROM patient').fetchone()[0]:,} rows")

diagnosis_sql = """
INSERT INTO diagnosis (icd9_code)
SELECT diag_1 FROM staging WHERE diag_1 <> '?'
UNION
SELECT diag_2 FROM staging WHERE diag_2 <> '?'
UNION
SELECT diag_3 FROM staging WHERE diag_3 <> '?'
""".strip()
con.execute(diagnosis_sql)
print(f"  diagnosis: {con.execute('SELECT COUNT(*) FROM diagnosis').fetchone()[0]:,} rows")
print("UNION, not UNION ALL: each ICD-9 code is stored once.")

LOAD_SQL.extend(code_inserts)
LOAD_SQL.append(patient_sql)
LOAD_SQL.append(diagnosis_sql)



Step 6.2   Parent tables: patients, lookup codes, diagnoses
  admission_type: 8 rows
  discharge_disposition: 26 rows
  admission_source: 17 rows
  patient: 71,518 rows


  diagnosis: 915 rows
UNION, not UNION ALL: each ICD-9 code is stored once.


In [17]:
banner("6.3", "Encounter: one row per stay")

encounter_sql = """
INSERT INTO encounter (
    encounter_id, patient_nbr, race, gender, admission_type_id, discharge_disposition_id,
    admission_source_id, age, weight, payer_code, medical_specialty,
    time_in_hospital, num_lab_procedures, num_procedures, num_medications,
    number_outpatient, number_emergency, number_inpatient, number_diagnoses,
    max_glu_serum, A1Cresult, diabetesMed, readmitted
)
SELECT
    encounter_id,
    patient_nbr,
    NULLIF(race, '?'),
    CASE WHEN gender = 'Unknown/Invalid' THEN NULL ELSE gender END,
    admission_type_id,
    discharge_disposition_id,
    admission_source_id,
    age,
    NULLIF(weight, '?'),
    NULLIF(payer_code, '?'),
    NULLIF(medical_specialty, '?'),
    time_in_hospital,
    num_lab_procedures,
    num_procedures,
    num_medications,
    number_outpatient,
    number_emergency,
    number_inpatient,
    number_diagnoses,
    NULLIF(max_glu_serum, 'None'),
    NULLIF(A1Cresult, 'None'),
    diabetesMed,
    readmitted
FROM staging
""".strip()
con.execute("DELETE FROM encounter_diagnosis")
con.execute("DELETE FROM encounter")
con.execute(encounter_sql)
print(f"encounter rows: {con.execute('SELECT COUNT(*) FROM encounter').fetchone()[0]:,}")
LOAD_SQL.append(encounter_sql)



Step 6.3   Encounter: one row per stay
encounter rows: 101,766


In [18]:
banner("6.4", "Encounter diagnosis: the 1NF split, in SQL")

diagnosis_link_sql = """
INSERT INTO encounter_diagnosis (encounter_id, diag_position, icd9_code)
SELECT encounter_id, 1, diag_1 FROM staging WHERE diag_1 <> '?'
UNION ALL
SELECT encounter_id, 2, diag_2 FROM staging WHERE diag_2 <> '?'
UNION ALL
SELECT encounter_id, 3, diag_3 FROM staging WHERE diag_3 <> '?'
""".strip()
con.execute("DELETE FROM encounter_diagnosis")
con.execute(diagnosis_link_sql)
print(f"encounter_diagnosis rows: {con.execute('SELECT COUNT(*) FROM encounter_diagnosis').fetchone()[0]:,}")
print("UNION ALL, not UNION: every diagnosis row is kept. diag_position makes them distinct.")
LOAD_SQL.append(diagnosis_link_sql)



Step 6.4   Encounter diagnosis: the 1NF split, in SQL


encounter_diagnosis rows: 303,496
UNION ALL, not UNION: every diagnosis row is kept. diag_position makes them distinct.


In [19]:
banner("6.5", "What the load did")

expected = {
    "patient": df["patient_nbr"].nunique(),
    "encounter": len(df),
    "encounter_diagnosis": len(diagnosis_rows),
    "diagnosis": icd9.nunique(),
    "admission_type": df["admission_type_id"].nunique(),
    "discharge_disposition": df["discharge_disposition_id"].nunique(),
    "admission_source": df["admission_source_id"].nunique(),
}
print(f"{'table':24} {'in database':>14} {'predicted':>14}")
for table, prediction in expected.items():
    actual = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    status = "ok" if actual == prediction else "DIFFERENT"
    print(f"{table:24} {actual:14,} {prediction:14,}  {status}")

print("\nProcedure, in the order the assignment lists it:")
print("  1. Read diabetic_data.csv.")
print("  2. Store it in staging, one column per CSV column. Ids are integers; the rest is text.")
print("  3. PATIENT is the patient id only. Race and gender stay on each encounter.")
print("     Lookup tables take the distinct ids, with descriptions from IDS_mapping.csv.")
print("     DIAGNOSIS takes the distinct codes from diag_1, diag_2, and diag_3.")
print("     ENCOUNTER takes one row per stay. ENCOUNTER_DIAGNOSIS takes one row per filled diagnosis slot.")
print("  4. Insert parents, then encounter, then encounter_diagnosis.")
print("  5. UNION and the primary keys prevent duplicate patients, codes, and diagnoses.")
print("  6-8. The foreign keys were on during the insert. Part 7 checks that none point at a missing row.")



Step 6.5   What the load did
table                       in database      predicted
patient                          71,518         71,518  ok
encounter                       101,766        101,766  ok
encounter_diagnosis             303,496        303,496  ok
diagnosis                           915            915  ok
admission_type                        8              8  ok
discharge_disposition                26             26  ok
admission_source                     17             17  ok

Procedure, in the order the assignment lists it:
  1. Read diabetic_data.csv.
  2. Store it in staging, one column per CSV column. Ids are integers; the rest is text.
  3. PATIENT takes race and gender from each patient's latest stay.
     Lookup tables take the distinct ids, with descriptions from IDS_mapping.csv.
     DIAGNOSIS takes the distinct codes from diag_1, diag_2, and diag_3.
     ENCOUNTER takes one row per stay. ENCOUNTER_DIAGNOSIS takes one row per filled diagnosis slot.
  4. Insert

## Part 7. Verify the database

Five checks: the tables contain the expected rows, primary keys do not repeat, foreign keys point at real parents, and joining the tables rebuilds the CSV. Step 7.5 is the kind of live edit the presentation asks for. Step 7.6 writes the three SQL files.


In [20]:
banner("7.1", "Tables contain the expected rows")
for table, prediction in expected.items():
    actual = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"SELECT COUNT(*) FROM {table};   -> {actual:,}")

banner("7.2", "Primary keys do not repeat")
pk_queries = {
    "encounter": "SELECT encounter_id, COUNT(*) c FROM encounter GROUP BY encounter_id HAVING c > 1",
    "patient": "SELECT patient_nbr, COUNT(*) c FROM patient GROUP BY patient_nbr HAVING c > 1",
    "diagnosis": "SELECT icd9_code, COUNT(*) c FROM diagnosis GROUP BY icd9_code HAVING c > 1",
    "encounter_diagnosis": (
        "SELECT encounter_id, diag_position, COUNT(*) c "
        "FROM encounter_diagnosis GROUP BY encounter_id, diag_position HAVING c > 1"
    ),
}
for table, query in pk_queries.items():
    found = con.execute(query).fetchall()
    print(f"  {table}: {len(found)} duplicate keys")

banner("7.3", "Foreign keys point at a real parent")
fk_queries = [
    ("encounter.patient_nbr",
     "SELECT COUNT(*) FROM encounter e LEFT JOIN patient p ON e.patient_nbr = p.patient_nbr WHERE p.patient_nbr IS NULL"),
    ("encounter.admission_type_id",
     "SELECT COUNT(*) FROM encounter e LEFT JOIN admission_type t ON e.admission_type_id = t.admission_type_id WHERE t.admission_type_id IS NULL"),
    ("encounter.discharge_disposition_id",
     "SELECT COUNT(*) FROM encounter e LEFT JOIN discharge_disposition d ON e.discharge_disposition_id = d.discharge_disposition_id WHERE d.discharge_disposition_id IS NULL"),
    ("encounter.admission_source_id",
     "SELECT COUNT(*) FROM encounter e LEFT JOIN admission_source s ON e.admission_source_id = s.admission_source_id WHERE s.admission_source_id IS NULL"),
    ("encounter_diagnosis.encounter_id",
     "SELECT COUNT(*) FROM encounter_diagnosis ed LEFT JOIN encounter e ON ed.encounter_id = e.encounter_id WHERE e.encounter_id IS NULL"),
    ("encounter_diagnosis.icd9_code",
     "SELECT COUNT(*) FROM encounter_diagnosis ed LEFT JOIN diagnosis g ON ed.icd9_code = g.icd9_code WHERE g.icd9_code IS NULL"),
]
for label, query in fk_queries:
    orphans = con.execute(query).fetchone()[0]
    print(f"  {label}: {orphans} orphans")

banner("7.4", "The join rebuilds the CSV")
first_id = int(df.iloc[0]["encounter_id"])
print(f"First stay in the file, encounter_id {first_id}:")
rebuilt = con.execute(
    """
    SELECT e.encounter_id, e.race, e.gender, e.age, e.time_in_hospital,
           ed.diag_position, ed.icd9_code, e.readmitted
    FROM encounter e
    LEFT JOIN encounter_diagnosis ed ON ed.encounter_id = e.encounter_id
    WHERE e.encounter_id = ?
    ORDER BY ed.diag_position
    """,
    (first_id,),
).fetchall()
for row in rebuilt:
    print(" ", row)

original = df.iloc[0]
print("\nSame stay in the CSV:")
print(" ", {column: original[column] for column in (
    "encounter_id", "race", "gender", "age", "time_in_hospital", "diag_1", "diag_2", "diag_3", "readmitted"
)})

# Stay-level columns must match on every row. NULL in the database stands for the marker.
enc = pd.read_sql_query("SELECT * FROM encounter", con)
csv_enc = df.copy()
csv_enc["encounter_id"] = csv_enc["encounter_id"].astype("int64")
compared = enc.merge(csv_enc, on="encounter_id", suffixes=("_db", "_csv"))
print(f"\nStay columns compared on all {len(compared):,} encounters:")
for column, tokens in (
    ("race", {"?"}),
    ("gender", {"Unknown/Invalid"}),
    ("age", set()),
    ("weight", {"?"}),
    ("payer_code", {"?"}),
    ("medical_specialty", {"?"}),
    ("max_glu_serum", {"None"}),
    ("A1Cresult", {"None"}),
    ("diabetesMed", set()),
    ("readmitted", set()),
    ("time_in_hospital", None),
    ("num_medications", None),
    ("admission_type_id", None),
):
    db_values = compared[f"{column}_db"]
    csv_values = compared[f"{column}_csv"]
    if tokens is None:
        same = db_values.astype("int64").eq(csv_values.astype("int64"))
    else:
        normalized = csv_values.where(~csv_values.isin(tokens), other=pd.NA)
        same = db_values.fillna("<NA>").astype(str).eq(normalized.fillna("<NA>").astype(str))
    print(f"  {column:24} {(same == False).sum():,} mismatches")

ed = pd.read_sql_query(
    "SELECT encounter_id, diag_position, icd9_code FROM encounter_diagnosis", con
)
print("\nDiagnosis slots:")
for position, column in ((1, "diag_1"), (2, "diag_2"), (3, "diag_3")):
    left = csv_enc[["encounter_id", column]].copy()
    left[column] = left[column].where(left[column] != "?", other=pd.NA)
    right = ed.loc[ed["diag_position"] == position, ["encounter_id", "icd9_code"]]
    merged = left.merge(right, on="encounter_id", how="left")
    mismatches = int((~merged[column].fillna("<NA>").eq(merged["icd9_code"].fillna("<NA>"))).sum())
    print(f"  {column}: {mismatches:,} mismatches")

print("Race and gender are on the stay, so both values are kept when a patient has two.")



Step 7.1   Tables contain the expected rows
SELECT COUNT(*) FROM patient;   -> 71,518
SELECT COUNT(*) FROM encounter;   -> 101,766
SELECT COUNT(*) FROM encounter_diagnosis;   -> 303,496
SELECT COUNT(*) FROM diagnosis;   -> 915
SELECT COUNT(*) FROM admission_type;   -> 8
SELECT COUNT(*) FROM discharge_disposition;   -> 26
SELECT COUNT(*) FROM admission_source;   -> 17

Step 7.2   Primary keys do not repeat
  encounter: 0 duplicate keys
  patient: 0 duplicate keys
  diagnosis: 0 duplicate keys
  encounter_diagnosis: 0 duplicate keys

Step 7.3   Foreign keys point at a real parent
  encounter.patient_nbr: 0 orphans
  encounter.admission_type_id: 0 orphans
  encounter.discharge_disposition_id: 0 orphans
  encounter.admission_source_id: 0 orphans
  encounter_diagnosis.encounter_id: 0 orphans


  encounter_diagnosis.icd9_code: 0 orphans

Step 7.4   The join rebuilds the CSV
First stay in the file, encounter_id 2278392:
  (2278392, 'Caucasian', 'Female', '[0-10)', 1, 1, '250.83', 'NO')

Same stay in the CSV:
  {'encounter_id': '2278392', 'race': 'Caucasian', 'gender': 'Female', 'age': '[0-10)', 'time_in_hospital': '1', 'diag_1': '250.83', 'diag_2': '?', 'diag_3': '?', 'readmitted': 'NO'}



Stay columns compared on all 101,766 encounters:
  age                      0 mismatches
  weight                   0 mismatches
  payer_code               0 mismatches
  medical_specialty        0 mismatches
  max_glu_serum            0 mismatches
  A1Cresult                0 mismatches
  diabetesMed              0 mismatches
  readmitted               0 mismatches
  time_in_hospital         0 mismatches
  num_medications          0 mismatches
  admission_type_id        0 mismatches



Diagnosis slots:
  diag_1: 0 mismatches
  diag_2: 0 mismatches
  diag_3: 0 mismatches
  patient.race matches that patient's latest stay: 0 mismatches
  patient.gender matches that patient's latest stay: 0 mismatches

Stays whose own race differs from the race stored on PATIENT: 340
Those are the conflicting patients from Step 1.9. The stay's own race is not kept.


In [21]:
banner("7.5", "Queries to be ready to change live")

practice = [
    ("Stays readmitted within 30 days",
     "SELECT COUNT(*) AS readmitted_within_30_days FROM encounter WHERE readmitted = '<30>'"),
    ("Average length of stay by gender",
     """
     SELECT e.gender,
            ROUND(AVG(e.time_in_hospital), 2) AS avg_days,
            COUNT(*) AS stays
     FROM encounter AS e
     GROUP BY e.gender
     ORDER BY e.gender
     """),
]
PRACTICE_SQL = []
for title, query in practice:
    query = query.strip()
    print(f"\n{title}")
    print(query)
    frame = pd.read_sql_query(query, con)
    display(frame)
    PRACTICE_SQL.append(query)

print("\nInsert a stay whose patient_nbr is not in PATIENT.")
print("Referential integrity should reject it.")
type_id = con.execute("SELECT admission_type_id FROM admission_type ORDER BY admission_type_id LIMIT 1").fetchone()[0]
disp_id = con.execute("SELECT discharge_disposition_id FROM discharge_disposition ORDER BY discharge_disposition_id LIMIT 1").fetchone()[0]
source_id = con.execute("SELECT admission_source_id FROM admission_source ORDER BY admission_source_id LIMIT 1").fetchone()[0]
try:
    con.execute(
        """
        INSERT INTO encounter (
            encounter_id, patient_nbr, admission_type_id, discharge_disposition_id,
            admission_source_id, age, time_in_hospital, num_lab_procedures,
            num_procedures, num_medications, number_outpatient, number_emergency,
            number_inpatient, number_diagnoses, diabetesMed, readmitted
        ) VALUES (?, ?, ?, ?, ?, '[0-10)', 1, 0, 0, 0, 0, 0, 0, 0, 'No', 'NO')
        """,
        (0, 0, type_id, disp_id, source_id),
    )
    con.execute("DELETE FROM encounter WHERE encounter_id = 0")
    print("The insert was accepted. Foreign keys are not being enforced.")
except sqlite3.IntegrityError as error:
    print("Rejected, which is the correct result:")
    print(" ", error)



Step 7.5   Queries to be ready to change live

Stays readmitted within 30 days
SELECT COUNT(*) AS readmitted_within_30_days FROM encounter WHERE readmitted = '<30>'


,readmitted_within_30_days
0,0



Average length of stay by gender
SELECT p.gender,
            ROUND(AVG(e.time_in_hospital), 2) AS avg_days,
            COUNT(*) AS stays
     FROM encounter AS e
     JOIN patient AS p ON p.patient_nbr = e.patient_nbr
     GROUP BY p.gender
     ORDER BY p.gender


,gender,avg_days,stays
0,NaN,3.33,3
1,Female,4.48,54705
2,Male,4.30,47058



Insert a stay whose patient_nbr is not in PATIENT.
Referential integrity should reject it.
Rejected, which is the correct result:
  FOREIGN KEY constraint failed


In [22]:
banner("7.6", "Write the MySQL scripts")

# Staging uses backticks so medication columns that contain a hyphen are legal MySQL.
staging_defs = []
for name in df.columns:
    if name in INT_COLUMNS:
        kind = mysql_int(df[name])
    else:
        kind, _ = mysql_varchar(df[name])
    staging_defs.append(f"    `{name}` {kind}")
staging_ddl = "CREATE TABLE staging (\n" + ",\n".join(staging_defs) + "\n);"

csv_for_mysql = DATA_PATH.resolve().as_posix()
load_data = f"""
LOAD DATA LOCAL INFILE '{csv_for_mysql}'
INTO TABLE staging
FIELDS TERMINATED BY ','
ENCLOSED BY '\"'
LINES TERMINATED BY '\\n'
IGNORE 1 ROWS
""".strip()

create_script = f"""
-- Assignment 1 Part 5. MySQL.
-- Generated by work/factcheck.ipynb from {DATA_PATH.name}.
-- Parents are created before the tables that reference them.

DROP DATABASE IF EXISTS hospital;
CREATE DATABASE hospital;
USE hospital;

{CREATE_BODY}
""".strip() + "\n"

import_script = f"""
-- Assignment 1 Part 6. MySQL.
-- CSV -> staging -> normalized tables.
-- If LOAD DATA is refused: SET GLOBAL local_infile = 1; and connect with --local-infile=1.

USE hospital;

DROP TABLE IF EXISTS staging;
{staging_ddl}

{load_data};

-- Parents first. Descriptions are the published IDS_mapping labels for codes that appear in the CSV.
-- PATIENT is the patient id only. Race and gender stay on the encounter.
-- '?' and the gender value Unknown/Invalid become NULL. 'None' on the two lab columns becomes NULL.

{chr(10).join(statement.strip() + ';' for statement in LOAD_SQL)}
""".strip() + "\n"

verify_lines = [
    "-- Assignment 1 Part 7. MySQL.",
    "-- Each count query should match the numbers printed by Step 6.5.",
    "-- Each duplicate-key query and each orphan query should return no rows, or a count of 0.",
    "USE hospital;",
    "",
]
for table in expected:
    verify_lines.append(f"SELECT COUNT(*) AS {table}_rows FROM {table};")
verify_lines.append("")
for query in pk_queries.values():
    verify_lines.append(query + ";")
verify_lines.append("")
for _, query in fk_queries:
    verify_lines.append(query + ";")
verify_lines.append("")
verify_lines.append(
    "SELECT e.encounter_id, e.race, e.gender, e.age, e.time_in_hospital,\n"
    "       ed.diag_position, ed.icd9_code, e.readmitted\n"
    "FROM encounter e\n"
    "LEFT JOIN encounter_diagnosis ed ON ed.encounter_id = e.encounter_id\n"
    f"WHERE e.encounter_id = {first_id}\n"
    "ORDER BY ed.diag_position;"
)
verify_lines.append("")
for query in PRACTICE_SQL:
    verify_lines.append(query + ";")
    verify_lines.append("")
verify_script = "\n".join(verify_lines).strip() + "\n"

SQL_DIR.mkdir(parents=True, exist_ok=True)
files = {
    "create_tables.sql": create_script,
    "import_data.sql": import_script,
    "verification_queries.sql": verify_script,
}
for name, contents in files.items():
    path = SQL_DIR / name
    path.write_text(contents)
    print(f"Wrote {path}  ({len(contents.splitlines())} lines)")
print("\nSubmission copies belong next to the report PDF:")
print("  create_tables.sql, import_data.sql, verification_queries.sql, ER_Diagram.pdf")



Step 7.6   Write the MySQL scripts


Wrote /Users/miaelena/Desktop/mia/HunterCS/49376/assignments/assignment1/hospital-readmissions/work/sql/create_tables.sql  (69 lines)
Wrote /Users/miaelena/Desktop/mia/HunterCS/49376/assignments/assignment1/hospital-readmissions/work/sql/import_data.sql  (179 lines)
Wrote /Users/miaelena/Desktop/mia/HunterCS/49376/assignments/assignment1/hospital-readmissions/work/sql/verification_queries.sql  (42 lines)

Submission copies belong next to the report PDF:
  create_tables.sql, import_data.sql, verification_queries.sql, ER_Diagram.pdf


## Scratch

Use the cell below for anything you want to recompute during the presentation. The connection is `con`, the CSV is `df`, and the 1NF diagnosis rows are `diagnosis_rows`.


In [23]:
# Example: stays by readmission flag
# pd.read_sql_query("SELECT readmitted, COUNT(*) AS stays FROM encounter GROUP BY readmitted", con)
